# Module 7: Agentic Workflows

**Goal:** Build progressively complex agent systems — from a single tool-calling agent up to a multi-agent LangGraph workflow.

**What you'll learn:**
1. What makes an LLM an "agent" (tool use + decision loop)
2. Defining tools with LangChain
3. Building a ReAct agent (Reason + Act)
4. State machines with LangGraph
5. Multi-agent routing (Researcher → Coder → Reviewer)
6. Human-in-the-loop checkpoints

---

## 0. Setup

In [ ]:
%pip install -q openai langchain langchain-core langchain-openai langchain-community langgraph pydantic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ Ready")

---
## 1. What Is an Agent?

A plain LLM call is **stateless**: question → answer, done.

An **agent** adds a loop:

```
User goal
   │
   ▼
[LLM] ──thinks──▶ "I need to call tool X"
   │                     │
   │◀── tool result ──────┘
   │
[LLM] ──thinks──▶ "I need tool Y too" → repeat
   │
[LLM] ──thinks──▶ "I have enough info now"
   │
Final answer
```

The LLM decides **which** tools to call, **when** to stop, and **how** to combine results. That decision-making capacity is what makes it an agent.

---
## 2. Defining Tools

In [ ]:
from langchain_core.tools import tool
import math, json, datetime

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Input must be a valid Python math expression.
    Examples: '2 + 2', 'math.sqrt(144)', '(10 * 5) / 2'
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_current_date(timezone: str = "UTC") -> str:
    """Get the current date and time. Timezone parameter is informational only."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + f" ({timezone})"

@tool
def word_count(text: str) -> str:
    """Count words, sentences, and characters in a text."""
    words = len(text.split())
    sentences = text.count('.') + text.count('!') + text.count('?')
    chars = len(text)
    return json.dumps({"words": words, "sentences": sentences, "characters": chars})

@tool
def search_knowledge_base(query: str) -> str:
    """Search the LLM engineering knowledge base. Returns relevant snippets."""
    # Simulated KB — in production this would hit a real vector store
    kb = {
        "rag": "RAG combines retrieval with generation. It retrieves relevant documents and uses them as context.",
        "lora": "LoRA fine-tunes models by adding small trainable matrices, reducing parameters by 99%+.",
        "agent": "Agents use LLMs as reasoning engines that decide which tools to call and when to stop.",
        "evaluation": "LLM evaluation uses automated metrics (ROUGE), LLM-as-judge, and human review.",
    }
    query_lower = query.lower()
    results = [v for k, v in kb.items() if k in query_lower]
    return results[0] if results else "No relevant information found in knowledge base."

tools = [calculator, get_current_date, word_count, search_knowledge_base]
print(f"Defined {len(tools)} tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

---
## 3. ReAct Agent (Reason + Act)

ReAct is the standard agent pattern: the LLM alternates between **reasoning** ("I should calculate X") and **acting** (calling a tool).

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate

REACT_TEMPLATE = """You are a helpful assistant with access to tools.
Use tools when needed to answer accurately.

Tools available:
{tools}

Format:
Question: the input question
Thought: what you need to do
Action: tool name (one of [{tool_names}])
Action Input: input for the tool
Observation: tool result
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information
Final Answer: the answer to the question

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(REACT_TEMPLATE)
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

# Test it
result = executor.invoke({"input": "What is the square root of 1764, and what is today's date?"})
print("\nFinal Answer:", result["output"])

In [ ]:
# Multi-step reasoning: requires combining tools
result = executor.invoke({
    "input": "Count the words in this sentence: 'The quick brown fox jumps over the lazy dog.' Then tell me what RAG stands for."
})
print("\nFinal Answer:", result["output"])

---
## 4. LangGraph State Machine

ReAct agents are a black box. **LangGraph** makes the agent's flow explicit — you define nodes (functions) and edges (routing rules) as a graph.

Benefits: inspectable, testable, supports cycles and human interrupts.

In [ ]:
from typing import TypedDict, Annotated, Literal
import operator
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# 1. Define shared state
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]  # append-only

# 2. Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# 3. Define nodes
def agent_node(state: AgentState) -> AgentState:
    """LLM decides: answer directly or call a tool."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)  # Executes whatever tool the LLM chose

# 4. Routing logic: continue looping or stop?
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"   # LLM wants to call a tool → loop back
    return "end"          # LLM gave a final answer → stop

# 5. Build graph
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
graph.add_edge("tools", "agent")  # After tool call → back to agent

app = graph.compile()
print("✅ Graph compiled")
print("Nodes:", list(app.get_graph().nodes.keys()))

In [ ]:
# Run the graph
result = app.invoke({"messages": [HumanMessage(content="What is 15% of 840? Show your calculation.")]})

print("Message trace:")
for msg in result["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    content = msg.content if msg.content else f"[tool call: {[tc['name'] for tc in msg.tool_calls]}]"
    print(f"  {role:12s}: {str(content)[:100]}")

---
## 5. Multi-Agent Routing

Different tasks need different specialists. A **router** inspects the request and dispatches to the right agent.

In [ ]:
from langchain_core.messages import SystemMessage

def make_specialist(role: str, expertise: str) -> callable:
    """Factory: create a specialist agent node."""
    system = SystemMessage(content=f"You are a {role}. {expertise} Be concise (2-3 sentences).")
    def node(state: AgentState) -> AgentState:
        print(f"  → Routing to: {role}")
        response = llm.invoke([system] + state["messages"])
        return {"messages": [AIMessage(content=f"[{role}] {response.content}")]}
    node.__name__ = role.lower().replace(" ", "_")
    return node

researcher = make_specialist("Research Analyst", "Explain concepts clearly with facts and examples.")
coder      = make_specialist("Senior Engineer",  "Give precise technical answers with code snippets.")
advisor    = make_specialist("Strategy Advisor", "Give practical, actionable business advice.")

def router(state: AgentState) -> Literal["researcher", "coder", "advisor"]:
    """Classify the query and route to the right specialist."""
    question = state["messages"][-1].content.lower()
    route_prompt = f"""Classify this question into exactly one category: 'research', 'code', or 'strategy'.
Output only the single word.
Question: {question}"""
    classification = llm.invoke([HumanMessage(content=route_prompt)]).content.strip().lower()
    mapping = {"research": "researcher", "code": "coder", "strategy": "advisor"}
    return mapping.get(classification, "researcher")

# Build multi-agent graph
multi = StateGraph(AgentState)
multi.add_node("router",     lambda s: s)   # Pass-through; routing happens in conditional edge
multi.add_node("researcher", researcher)
multi.add_node("coder",      coder)
multi.add_node("advisor",    advisor)
multi.set_entry_point("router")
multi.add_conditional_edges("router", router, {"researcher": "researcher", "coder": "coder", "advisor": "advisor"})
multi.add_edge("researcher", END)
multi.add_edge("coder",      END)
multi.add_edge("advisor",    END)
multi_app = multi.compile()

test_questions = [
    "What is transformer attention and how does it work?",
    "Write a Python function to chunk text into overlapping windows.",
    "Should I fine-tune my own model or use an API for a customer support bot?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    result = multi_app.invoke({"messages": [HumanMessage(content=q)]})
    print(f"A: {result['messages'][-1].content[:200]}...")

---
## 6. Human-in-the-Loop

Some decisions are too consequential for full automation. LangGraph supports **interrupt points** — the graph pauses, presents context to a human, and resumes with their input.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt
from typing import TypedDict, Annotated
import operator

class ReviewState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    draft: str
    approved: bool

def draft_node(state: ReviewState) -> ReviewState:
    """LLM drafts a response."""
    question = state["messages"][-1].content
    draft = llm.invoke([HumanMessage(content=f"Draft a response to: {question}")]).content
    print(f"\n📝 Draft ready: {draft[:150]}...")
    return {"draft": draft}

def review_node(state: ReviewState) -> ReviewState:
    """Human reviews the draft — interrupt point."""
    # In a real app, this would pause and send the draft to a UI
    # For demo: auto-approve if draft looks reasonable
    approved = len(state["draft"]) > 50  # simplified approval logic
    print(f"\n👤 Human review: {'✅ Approved' if approved else '❌ Rejected'}")
    return {"approved": approved}

def finalize_node(state: ReviewState) -> ReviewState:
    """Send approved response or request revision."""
    if state["approved"]:
        msg = AIMessage(content=state["draft"])
    else:
        msg = AIMessage(content="Draft rejected. Please rephrase your question.")
    return {"messages": [msg]}

hitl = StateGraph(ReviewState)
hitl.add_node("draft",    draft_node)
hitl.add_node("review",   review_node)
hitl.add_node("finalize", finalize_node)
hitl.set_entry_point("draft")
hitl.add_edge("draft",    "review")
hitl.add_edge("review",   "finalize")
hitl.add_edge("finalize", END)
hitl_app = hitl.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "review-session-1"}}
result = hitl_app.invoke(
    {"messages": [HumanMessage(content="Explain quantum entanglement")], "draft": "", "approved": False},
    config=config
)
print(f"\nFinal: {result['messages'][-1].content[:200]}")

---
## 7. Agent Design Patterns Cheat Sheet

| Pattern | When to use | Trade-off |
|---------|-------------|----------|
| **ReAct** | General tool use | Simple but hard to control |
| **LangGraph** | Complex flows, loops, human review | More setup, fully inspectable |
| **Multi-agent router** | Diverse task types | Routing overhead, need good classifier |
| **Hierarchical** | Large projects (manager → sub-agents) | High token cost |
| **Parallel agents** | Independent sub-tasks | Need result aggregation |

## 🧪 Exercises

1. **Add a tool**: Add a `unit_converter` tool (e.g., km↔miles, °C↔°F). Verify the agent uses it automatically when asked a conversion question.
2. **Tool failure handling**: Make `calculator` raise an exception for division by zero. What does the agent do? How can you make it recover gracefully?
3. **Improve the router**: The current router uses an LLM classification call. Replace it with keyword matching. Which approach is faster? Which is more accurate?
4. **Cycle limit**: Ask the agent a question it can't answer with available tools (e.g., "What's the weather in Tokyo?"). What happens? Add a `max_iterations` guard.
5. **State inspection**: Add a `step_count` field to `AgentState`. Increment it in each node. Use it to enforce a hard limit and emit a warning.

---
**Next:** [Module 08 — LLMOps & Observability](../08-llmops-observability/llmops_observability.ipynb)